In [ ]:
prefix_path = '../..'
import sys
sys.path.append(prefix_path)

import logging
import json
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from detection_baseline.detection_halp import load_halp_dataset, halp_collate_fn, HALPMLP, train_halp_detector, eval_halp_detector
from egh_vlm.utils import load_phd_dataset, split_stratified

In [ ]:
DATASET_PATH = f'{prefix_path}/data/phd/phd_base_qwen3.json'
IMG_FOLDER_PATH = f'{prefix_path}/data/phd/images'
FEATURES_PATH = f'{prefix_path}/data/phd/features/features_halp_phd_base_qwen3.pt'

FEATURE_KEYS = ['vf', 'vt', 'qt']
DETECTOR_PATHS = {
    key: f'{prefix_path}/data/phd/detectors/mlp_halp_{key}_phd_base_qwen3.pt'
    for key in FEATURE_KEYS
}
OUTPUT_PATHS = {
    key: f'{prefix_path}/data/phd/evaluations/evaluation_halp_mlp_{key}_phd_base_qwen3.json'
    for key in FEATURE_KEYS
}
LOGGING_PATHS = {
    key: f'{prefix_path}/data/logs/log_halp_mlp_{key}_phd_base_qwen3.txt'
    for key in FEATURE_KEYS
}

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
TRAIN_RATIO = 0.7

#### Load Dataset

In [ ]:
dataset = load_phd_dataset(
    dataset_path=DATASET_PATH,
    img_folder_path=IMG_FOLDER_PATH,
)
print(f'Length of dataset: {len(dataset)}')

features = load_halp_dataset(FEATURES_PATH)
print(f'Length of features: {len(features)}')

hidden_size = features[0][1].size(-1) if len(features) > 0 else 0
print(f'Hidden size of features: {hidden_size}')

#### Experiment

In [ ]:
train_dataset, test_dataset = split_stratified(features, train_ratio=TRAIN_RATIO, random_state=42)
train_dataloader = DataLoader(
    train_dataset,
    batch_size=32,
    collate_fn=halp_collate_fn,
    shuffle=True,
 )
test_dataloader = DataLoader(
    test_dataset,
    batch_size=32,
    collate_fn=halp_collate_fn,
    shuffle=True,
 )

In [ ]:
def train_feature(
    feature_key: str,
    train_dataloader: DataLoader,
    test_dataloader: DataLoader,
    input_dim: int,
    device: torch.device,
    logging_path: str,
    detector_path: str,
    output_path: str,
    train_ratio: float = TRAIN_RATIO,
    epochs_count: int = 50,
    lr: float = 1e-3,
    dtype: torch.dtype = torch.float32,
    seed: int = 42,
 ):
    logging.basicConfig(filename=logging_path, level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
    logging.info(f'Start training feature={feature_key}, input_dim={input_dim}, train_ratio={train_ratio}, epochs={epochs_count}, lr={lr}')

    result = {
        'feature_key': feature_key,
        'train_ratio': train_ratio,
        'epochs_count': epochs_count,
        'lr': lr,
        'acc': {'value': 0.0, 'epoch': -1},
        'f1': {'value': 0.0, 'epoch': -1},
        'pr_auc': {'value': 0.0, 'epoch': -1},
    }
    torch.manual_seed(seed)
    detector = HALPMLP(input_dim=input_dim, device=device, dtype=dtype)
    loss_fn = nn.BCELoss()
    optim = torch.optim.Adam(detector.parameters(), lr=lr)

    for epoch in range(epochs_count):
        detector.train()
        for _, vfs, vts, qts, labels in train_dataloader:
            if feature_key == 'vf':
                feats = vfs
            elif feature_key == 'vt':
                feats = vts
            elif feature_key == 'qt':
                feats = qts
            else:
                raise ValueError(f'Unknown feature_key: {feature_key!r}. Use vf, vt, or qt.')

            optim.zero_grad()
            output = detector(feats).squeeze(-1)
            label = labels.to(device=output.device, dtype=output.dtype)
            output = torch.nan_to_num(output, nan=0.0, posinf=1.0, neginf=0.0).clamp(1e-6, 1.0 - 1e-6)
            loss = loss_fn(output, label.clamp(0.0, 1.0))
            loss.backward()
            optim.step()

        metrics = eval_halp_detector(detector, test_dataloader, feature_key)
        acc = metrics['acc']
        f1 = metrics['f1']
        pr_auc = metrics['pr_auc']

        logging.info(f'Epoch {epoch + 1}/{epochs_count}, Acc: {acc:.4f}, F1: {f1:.4f}, PR-AUC: {pr_auc:.4f}')
        print(f'[{feature_key}] Epoch {epoch + 1}/{epochs_count}, Acc: {acc:.4f}, F1: {f1:.4f}, PR-AUC: {pr_auc:.4f}')

        if acc > result['acc']['value']:
            result['acc'] = {'value': acc, 'epoch': epoch + 1}
        if f1 > result['f1']['value']:
            result['f1'] = {'value': f1, 'epoch': epoch + 1}
        if pr_auc > result['pr_auc']['value']:
            result['pr_auc'] = {'value': pr_auc, 'epoch': epoch + 1}
            torch.save(detector.state_dict(), detector_path)

    with open(output_path, 'w') as f:
        json.dump(result, f, indent=2)

    logging.info("Best Accuracy: %.4f at epoch %s", result['acc']['value'], result['acc']['epoch'])
    logging.info("Best F1: %.4f at epoch %s", result['f1']['value'], result['f1']['epoch'])
    logging.info("Best PR-AUC: %.4f at epoch %s", result['pr_auc']['value'], result['pr_auc']['epoch'])
    print(f"[{feature_key}] Best Accuracy: {result['acc']['value']:.4f} at epoch {result['acc']['epoch']}")
    print(f"[{feature_key}] Best F1: {result['f1']['value']:.4f} at epoch {result['f1']['epoch']}")
    print(f"[{feature_key}] Best PR-AUC: {result['pr_auc']['value']:.4f} at epoch {result['pr_auc']['epoch']}")
    print(f'Finished training feature={feature_key}.\n')
    
    logger = logging.getLogger()
    for handler in logger.handlers[:]:
        handler.close()
        logger.removeHandler(handler)

    return detector, result

In [ ]:
results = {}
for feature_key in FEATURE_KEYS:
    detector_path = DETECTOR_PATHS[feature_key]
    output_path = OUTPUT_PATHS[feature_key]
    logging_path = LOGGING_PATHS[feature_key]

    detector, result = train_feature(
        feature_key=feature_key,
        train_dataloader=train_dataloader,
        test_dataloader=test_dataloader,
        input_dim=hidden_size,
        device=DEVICE,
        logging_path=logging_path,
        detector_path=detector_path,
        output_path=output_path,
    )
    results[feature_key] = result